In [1]:
import pandas as pd
import os
from pathlib import Path
import re
from typing import List, Dict, Tuple

# Install required packages if not already installed
try:
    import PyPDF2
except ImportError:
    print("Installing PyPDF2...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "PyPDF2"])
    import PyPDF2

print("All required packages are available!")

def extract_pdf_text_with_pages(pdf_path: str) -> Tuple[str, Dict[int, str]]:
    """Extract text from PDF file and return both combined text and page-wise text."""
    try:
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            full_text = ""
            page_texts = {}
            
            for page_num, page in enumerate(pdf_reader.pages, 1):
                page_text = page.extract_text()
                page_texts[page_num] = page_text.lower()
                full_text += page_text
                
        return full_text.lower(), page_texts
    except Exception as e:
        print(f"Error reading PDF: {e}")
        return "", {}

def find_requirement_pages(requirement: str, page_texts: Dict[int, str]) -> List[int]:
    """Find which pages contain the requirement text."""
    requirement_lower = requirement.lower()
    found_pages = []
    
    for page_num, page_text in page_texts.items():
        if requirement_lower in page_text:
            found_pages.append(page_num)
    
    return found_pages

def extract_matching_snippets(requirement: str, page_texts: Dict[int, str], context_chars: int = 200) -> Dict[int, List[str]]:
    """Extract text snippets from pages that contain the requirement."""
    requirement_lower = requirement.lower()
    page_snippets = {}
    
    for page_num, page_text in page_texts.items():
        if requirement_lower in page_text:
            snippets = []
            start = 0
            while True:
                # Find the next occurrence of the requirement
                pos = page_text.find(requirement_lower, start)
                if pos == -1:
                    break
                
                # Extract context around the match
                snippet_start = max(0, pos - context_chars)
                snippet_end = min(len(page_text), pos + len(requirement_lower) + context_chars)
                snippet = page_text[snippet_start:snippet_end]
                
                # Clean up the snippet
                snippet = ' '.join(snippet.split())  # Remove extra whitespace
                
                # Add ellipsis if we truncated
                if snippet_start > 0:
                    snippet = "..." + snippet
                if snippet_end < len(page_text):
                    snippet = snippet + "..."
                
                snippets.append(snippet)
                start = pos + 1
            
            if snippets:
                page_snippets[page_num] = snippets
    
    return page_snippets

def check_requirement_in_text(requirement: str, pdf_text: str, page_texts: Dict[int, str]) -> Dict:
    """Check if requirement appears in PDF text and find pages with text snippets."""
    requirement_lower = requirement.lower()
    
    # Direct exact match
    exact_match = requirement_lower in pdf_text
    exact_pages = find_requirement_pages(requirement, page_texts) if exact_match else []
    exact_snippets = extract_matching_snippets(requirement, page_texts) if exact_match else {}
    
    return {
        'exact_match': exact_match,
        'exact_pages': exact_pages,
        'exact_snippets': exact_snippets
    }

def check_document_requirements(document_id: int, csv_path: str, documents_folder: str) -> pd.DataFrame:
    """Check all requirements for a specific document_id against the PDF."""
    
    # Read CSV file
    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"Error reading CSV file: {e}")
        return pd.DataFrame()
    
    # Filter for specific document_id
    doc_requirements = df[df['document_id'] == document_id].copy()
    
    if doc_requirements.empty:
        print(f"No requirements found for document_id {document_id}")
        return pd.DataFrame()
    
    # Find PDF file
    pdf_filename = f"{document_id}.pdf"
    pdf_path = os.path.join(documents_folder, pdf_filename)
    
    if not os.path.exists(pdf_path):
        print(f"PDF file not found: {pdf_path}")
        return pd.DataFrame()
    
    print(f"Processing document {document_id}...")
    print(f"Found {len(doc_requirements)} requirements to check")
    
    # Extract PDF text with page information
    pdf_text, page_texts = extract_pdf_text_with_pages(pdf_path)
    if not pdf_text:
        print("Could not extract text from PDF")
        return pd.DataFrame()
    
    print(f"Extracted {len(pdf_text)} characters from {len(page_texts)} pages")
    
    # Check each requirement
    results = []
    for idx, row in doc_requirements.iterrows():
        requirement = str(row['requirement'])
        if pd.isna(requirement) or requirement.strip() == '':
            continue
            
        check_result = check_requirement_in_text(requirement, pdf_text, page_texts)
        
        # Format snippets for storage
        snippets_text = ""
        for page_num in sorted(check_result['exact_snippets'].keys()):
            snippets_text += f"Page {page_num}: "
            for i, snippet in enumerate(check_result['exact_snippets'][page_num]):
                if i > 0:
                    snippets_text += " | "
                snippets_text += f'"{snippet}"'
            snippets_text += "\n"
        
        results.append({
            'row_index': idx,
            'document_id': document_id,
            'model_name': row.get('model_name', ''),
            'constraint_type': row.get('constraint_type', ''),
            'scope': row.get('scope', ''),
            'numerical_value': row.get('numerical_value', ''),
            'unit': row.get('unit', ''),
            'requirement': requirement,
            'exact_match': check_result['exact_match'],
            'pages_found': ', '.join(map(str, check_result['exact_pages'])) if check_result['exact_pages'] else '',
            'matching_text_snippets': snippets_text.strip()
        })
    
    return pd.DataFrame(results)

# Main execution
if __name__ == "__main__":
    # Configuration
    DOCUMENT_ID = 26
    CSV_PATH = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Extract_regulations/Filtered_regulations/combined_all_filtered_constraints.csv"
    DOCUMENTS_FOLDER = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Documents"
    OUTPUT_FOLDER = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Extract_regulations/Check_doc_result"
    
    # Create output folder if it doesn't exist
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    
    print(f"Checking requirements for document ID: {DOCUMENT_ID}")
    print(f"CSV path: {CSV_PATH}")
    print(f"Documents folder: {DOCUMENTS_FOLDER}")
    print(f"Output folder: {OUTPUT_FOLDER}")
    
    # Check requirements
    results_df = check_document_requirements(DOCUMENT_ID, CSV_PATH, DOCUMENTS_FOLDER)
    
    if not results_df.empty:
        # Display summary
        total_requirements = len(results_df)
        exact_matches = results_df['exact_match'].sum()
        
        print(f"\n=== SUMMARY FOR DOCUMENT {DOCUMENT_ID} ===")
        print(f"Total requirements checked: {total_requirements}")
        print(f"Exact matches found: {exact_matches}")
        print(f"Not found: {total_requirements - exact_matches}")
        print(f"Success rate: {exact_matches/total_requirements*100:.1f}%")
        
        # Display detailed results with page information and text snippets
        print(f"\n=== DETAILED RESULTS ===")
        for idx, row in results_df.iterrows():
            status = "✓ FOUND" if row['exact_match'] else "✗ NOT FOUND"
            pages_info = f" on pages: {row['pages_found']}" if row['pages_found'] else ""
            
            print(f"\n{status}{pages_info}")
            print(f"  Requirement: {row['requirement'][:100]}...")
            print(f"  Type: {row['constraint_type']}")
            print(f"  Scope: {row['scope']}")
            if row['numerical_value']:
                print(f"  Value: {row['numerical_value']} {row['unit']}")
            
            # Display matching text snippets
            if row['matching_text_snippets']:
                print(f"  MATCHING TEXT FROM PDF:")
                for line in row['matching_text_snippets'].split('\n'):
                    if line.strip():
                        print(f"    {line}")
        
        # Save results to CSV in specified folder
        output_filename = os.path.join(OUTPUT_FOLDER, f"document_{DOCUMENT_ID}_requirement_check.csv")
        results_df.to_csv(output_filename, index=False)
        print(f"\nResults saved to: {output_filename}")
        
        # Save detailed summary report
        summary_filename = os.path.join(OUTPUT_FOLDER, f"document_{DOCUMENT_ID}_summary_report.txt")
        with open(summary_filename, 'w', encoding='utf-8') as f:
            f.write(f"REQUIREMENT CHECK REPORT FOR DOCUMENT {DOCUMENT_ID}\n")
            f.write("=" * 60 + "\n\n")
            f.write(f"Total requirements checked: {total_requirements}\n")
            f.write(f"Exact matches found: {exact_matches}\n")
            f.write(f"Not found: {total_requirements - exact_matches}\n")
            f.write(f"Success rate: {exact_matches/total_requirements*100:.1f}%\n\n")
            
            f.write("DETAILED RESULTS:\n")
            f.write("-" * 40 + "\n\n")
            
            for idx, row in results_df.iterrows():
                status = "✓ FOUND" if row['exact_match'] else "✗ NOT FOUND"
                pages_info = f" on pages: {row['pages_found']}" if row['pages_found'] else ""
                
                f.write(f"{status}{pages_info}\n")
                f.write(f"  Requirement: {row['requirement']}\n")
                f.write(f"  Type: {row['constraint_type']}\n")
                f.write(f"  Scope: {row['scope']}\n")
                if row['numerical_value']:
                    f.write(f"  Value: {row['numerical_value']} {row['unit']}\n")
                
                # Write matching text snippets
                if row['matching_text_snippets']:
                    f.write(f"  MATCHING TEXT FROM PDF:\n")
                    for line in row['matching_text_snippets'].split('\n'):
                        if line.strip():
                            f.write(f"    {line}\n")
                
                f.write("\n")
        
        # Show requirements not found
        not_found = results_df[~results_df['exact_match']]
        if not not_found.empty:
            print(f"\n=== REQUIREMENTS NOT FOUND ({len(not_found)}) ===")
            not_found_filename = os.path.join(OUTPUT_FOLDER, f"document_{DOCUMENT_ID}_not_found.txt")
            with open(not_found_filename, 'w', encoding='utf-8') as f:
                f.write(f"REQUIREMENTS NOT FOUND IN DOCUMENT {DOCUMENT_ID}\n")
                f.write("=" * 50 + "\n\n")
                for idx, row in not_found.iterrows():
                    requirement_text = f"- {row['requirement']}"
                    print(requirement_text)
                    f.write(requirement_text + "\n")
            print(f"\nNot found requirements saved to: {not_found_filename}")
    else:
        print("No results to display.")
        
    print(f"\nAll output files saved to: {OUTPUT_FOLDER}")

unknown widths : 
[0, IndirectObject(10107, 0, 5085209936)]
unknown widths : 
[0, IndirectObject(10111, 0, 5085209936)]
unknown widths : 
[0, IndirectObject(14404, 0, 5085209936)]
unknown widths : 
[0, IndirectObject(14394, 0, 5085209936)]
unknown widths : 
[0, IndirectObject(14399, 0, 5085209936)]
unknown widths : 
[0, IndirectObject(14387, 0, 5085209936)]
unknown widths : 
[0, IndirectObject(14382, 0, 5085209936)]
unknown widths : 
[0, IndirectObject(1043, 0, 5085209936)]
unknown widths : 
[0, IndirectObject(1046, 0, 5085209936)]
unknown widths : 
[0, IndirectObject(1049, 0, 5085209936)]
unknown widths : 
[0, IndirectObject(1052, 0, 5085209936)]
unknown widths : 
[0, IndirectObject(1055, 0, 5085209936)]
unknown widths : 
[0, IndirectObject(1058, 0, 5085209936)]
unknown widths : 
[0, IndirectObject(1061, 0, 5085209936)]
unknown widths : 
[0, IndirectObject(1064, 0, 5085209936)]
unknown widths : 
[0, IndirectObject(1067, 0, 5085209936)]
unknown widths : 
[0, IndirectObject(1070, 0, 508

All required packages are available!
Checking requirements for document ID: 26
CSV path: /Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Extract_regulations/Filtered_regulations/combined_all_filtered_constraints.csv
Documents folder: /Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Documents
Output folder: /Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Extract_regulations/Check_doc_result
Processing document 26...
Found 504 requirements to check


unknown widths : 
[0, IndirectObject(1151, 0, 5085209936)]
unknown widths : 
[0, IndirectObject(1100, 0, 5085209936)]
unknown widths : 
[0, IndirectObject(1109, 0, 5085209936)]
unknown widths : 
[0, IndirectObject(1154, 0, 5085209936)]
unknown widths : 
[0, IndirectObject(1112, 0, 5085209936)]
unknown widths : 
[0, IndirectObject(1115, 0, 5085209936)]
unknown widths : 
[0, IndirectObject(1118, 0, 5085209936)]
unknown widths : 
[0, IndirectObject(1121, 0, 5085209936)]
unknown widths : 
[0, IndirectObject(1124, 0, 5085209936)]
unknown widths : 
[0, IndirectObject(1127, 0, 5085209936)]
unknown widths : 
[0, IndirectObject(1130, 0, 5085209936)]
unknown widths : 
[0, IndirectObject(1103, 0, 5085209936)]
unknown widths : 
[0, IndirectObject(1136, 0, 5085209936)]
unknown widths : 
[0, IndirectObject(1133, 0, 5085209936)]
unknown widths : 
[0, IndirectObject(1139, 0, 5085209936)]
unknown widths : 
[0, IndirectObject(1142, 0, 5085209936)]
unknown widths : 
[0, IndirectObject(1145, 0, 5085209936

Extracted 506556 characters from 177 pages

=== SUMMARY FOR DOCUMENT 26 ===
Total requirements checked: 504
Exact matches found: 84
Not found: 420
Success rate: 16.7%

=== DETAILED RESULTS ===

✗ NOT FOUND
  Requirement: export cables within an export cable corridor of up to 35 km (21.7 mi) in length on the OCS...
  Type: Spatial
  Scope: OCS export cable corridor
  Value: 35 km

✗ NOT FOUND
  Requirement: The use of cable protection measures must not exceed 10 percent of the total export and inter-array ...
  Type: Technical
  Scope: Cable protection measures
  Value: 10 percent

✗ NOT FOUND
  Requirement: The Lessee may construct and install on the Outer Continental Shelf (OCS) up to 114 wind turbine gen...
  Type: Spatial
  Scope: Outer Continental Shelf (OCS)
  Value: 114 wind turbine generators

✗ NOT FOUND
  Requirement: In areas where the final cable burial depth is less than 1.0 m below seabed, excluding cable crossin...
  Type: Technical
  Scope: Cable burial depth and protect